# Feed-forward Neural Network

BLEU moyen: 0.023595829946816325

ROUGE-L moyen: 0.09428864264584307

BERTScore moyen: 0.7642185151576996


In [1]:
import pandas as pd
import numpy as np
import re
import string
import tensorflow as tf
from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge
import evaluate
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Flatten, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.sequence import pad_sequences

2025-05-07 00:00:45.374045: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746568845.784659    1506 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746568845.824437    1506 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1746568846.241891    1506 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1746568846.241920    1506 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1746568846.241921    1506 computation_placer.cc:177] computation placer alr

In [2]:
df = pd.read_csv("MovieDataThread.csv")
df = df.dropna(subset=["Script"])
df = df.rename(columns={"Script": "text"})
df = df.sample(n=1000, random_state=42)


In [3]:
import re
from nltk.tokenize import word_tokenize

def script_tokenize(text):
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"\n", " NEWLINE_TOKEN ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip().split()

def custom_tokenize(text):
    first_tokenize = script_tokenize(text)
    entokenized = " ".join(first_tokenize)

    second_tokenize = word_tokenize(entokenized)
    return second_tokenize

In [4]:
tokenized_texts = [custom_tokenize(text) for text in df["text"]]
w2v_model = Word2Vec(sentences=tokenized_texts, vector_size=100, window=5, min_count=2, workers=4)

word_to_idx = {"<PAD>": 0, "<UNK>": 1}
for i, word in enumerate(w2v_model.wv.index_to_key):
    word_to_idx[word] = i + 2
idx_to_word = {i: w for w, i in word_to_idx.items()}

embedding_matrix = np.zeros((len(word_to_idx), 100))
for word, i in word_to_idx.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]

def encode_sequence(tokens):
    return [word_to_idx.get(t, 1) for t in tokens]

def prepare_sequences(tokenized_texts, seq_len=20):
    X, y = [], []
    for tokens in tokenized_texts:
        encoded = encode_sequence(tokens)
        for i in range(len(encoded) - seq_len):
            X.append(encoded[i:i+seq_len])
            y.append(encoded[i+seq_len])
    return np.array(X), np.array(y)

X, y = prepare_sequences(tokenized_texts, seq_len=20)
X = X[:50000]
y = y[:50000]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [5]:
model = Sequential([
    Embedding(
        input_dim=len(word_to_idx),
        output_dim=100,
        weights=[embedding_matrix],
        input_length=20,
        trainable=False
    ),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(len(word_to_idx), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stopping = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

/home/rickgao/miniconda3/envs/tf-wsl/lib/python3.9/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
W0000 00:00:1746569045.483923    1506 gpu_device.cc:2341] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [6]:
model.fit(X_train, y_train, validation_split=0.1, batch_size=64, epochs=20, callbacks=[early_stopping])

model.save("fnn_model.h5")

Epoch 1/20
563/563 ━━━━━━━━━━━━━━━━━━━━ 75s 131ms/step - accuracy: 0.1394 - loss: 6.4769 - val_accuracy: 0.2157 - val_loss: 5.3280
Epoch 2/20
563/563 ━━━━━━━━━━━━━━━━━━━━ 75s 134ms/step - accuracy: 0.2196 - loss: 4.9513 - val_accuracy: 0.2280 - val_loss: 5.1868
Epoch 3/20
563/563 ━━━━━━━━━━━━━━━━━━━━ 72s 129ms/step - accuracy: 0.2448 - loss: 4.4963 - val_accuracy: 0.2400 - val_loss: 5.1393
Epoch 4/20
563/563 ━━━━━━━━━━━━━━━━━━━━ 71s 126ms/step - accuracy: 0.2584 - loss: 4.2110 - val_accuracy: 0.2390 - val_loss: 5.2825
Epoch 5/20
563/563 ━━━━━━━━━━━━━━━━━━━━ 72s 127ms/step - accuracy: 0.2727 - loss: 3.9597 - val_accuracy: 0.2463 - val_loss: 5.3251
Epoch 6/20
563/563 ━━━━━━━━━━━━━━━━━━━━ 73s 129ms/step - accuracy: 0.2836 - loss: 3.7259 - val_accuracy: 0.2445 - val_loss: 5.5058


In [7]:
def generate_text(model, seed_text, word_to_idx, idx_to_word, max_len=50, seq_len=20, temperature=1.0):
    tokens = custom_tokenize(seed_text)
    for _ in range(max_len):
        context = tokens[-seq_len:]
        input_indices = [word_to_idx.get(w, word_to_idx["<UNK>"]) for w in context]
        if len(input_indices) < seq_len:
            input_indices = [word_to_idx["<PAD>"]] * (seq_len - len(input_indices)) + input_indices
        input_array = np.array(input_indices)[None, :]
        preds = model.predict(input_array, verbose=0)[0]
        preds = np.log(preds + 1e-8) / temperature
        preds = np.exp(preds) / np.sum(np.exp(preds))
        next_idx = np.random.choice(len(preds), p=preds)
        next_word = idx_to_word.get(next_idx, "<UNK>")
        if next_word == "<PAD>":
            break
        tokens.append(next_word)
    return " ".join(tokens)

In [ ]:
bleu = evaluate.load("bleu")
rouge = Rouge()
bertscore = evaluate.load("bertscore")

test_texts = df["text"].tolist()[-200:]

def calculate_bleu(reference, candidate):
    return sentence_bleu([reference], candidate, smoothing_function=SmoothingFunction().method4)

def calculate_rouge(reference, candidate):
    return rouge.get_scores(candidate, reference)[0]['rouge-l']['f']

print("\n===== Évaluation sur 10 exemples =====")
bleu_scores, rouge_scores, bert_scores = [], [], []

for full in test_texts:
    prompt = full[:150]
    reference = full[150:250]
    generated = generate_text(model, prompt, word_to_idx, idx_to_word, max_len=100)

    ref_tokens = custom_tokenize(reference)
    gen_tokens = custom_tokenize(generated)

    bleu_scores.append(calculate_bleu(ref_tokens, gen_tokens))
    rouge_scores.append(calculate_rouge(reference, generated))
    bert_result = bertscore.compute(predictions=[generated], references=[reference], lang="en")
    bert_scores.append(bert_result["f1"][0])

print("BLEU moyen:", sum(bleu_scores) / len(bleu_scores))
print("ROUGE-L moyen:", sum(rouge_scores) / len(rouge_scores))
print("BERTScore moyen:", sum(bert_scores) / len(bert_scores))


===== Évaluation sur 10 exemples =====


/home/rickgao/miniconda3/envs/tf-wsl/lib/python3.9/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BLEU moyen: 0.023595829946816325
ROUGE-L moyen: 0.09428864264584307
BERTScore moyen: 0.7642185151576996
